# Step 2 — Knowledge Wiki (Qdrant, configurable transport)

Ingest transcript/filing chunks into **Qdrant** and retrieve with citations
(`docs/wiki-ingestion.md`). The transport is **configurable** via `.env`:

- `QDRANT_MODE=memory` — in-process, zero setup (default)
- `QDRANT_MODE=docker` — remote server: `docker run -p 6333:6333 qdrant/qdrant`
- `QDRANT_MODE=local`  — on-disk persistence under `QDRANT_PATH`

Embeddings are configurable too (`EMBEDDING_BACKEND`: `hash` offline / `sentence-transformers` /
`vllm`). Dependencies come from `requirements.txt`.

In [1]:
import sys
from pathlib import Path
def _root():
    p = Path.cwd()
    for d in (p, *p.parents):
        if (d / "requirements.txt").exists():
            return d
    return p
ROOT = _root()
sys.path.insert(0, str(ROOT / "src"))
from ir_copilot.config import settings
print("backends -> qdrant:", settings.qdrant_mode, "| embeddings:", settings.embedding_backend,
      "| sentiment:", settings.sentiment_backend, "| llm:", settings.llm_backend)

backends -> qdrant: memory | embeddings: hash | sentiment: lexicon | llm: mock


In [2]:
from ir_copilot.embeddings import get_embedder
from ir_copilot.vectorstore import WikiStore, WikiChunk
from ir_copilot import corpus

embedder = get_embedder()
print(f"embedder: {type(embedder).__name__}  dim={embedder.dim}")

wiki = WikiStore(embedder)
wiki.ensure_collection(recreate=True)
chunks = [WikiChunk(chunk_id=str(i), **c) for i, c in enumerate(corpus.TRANSCRIPT_CHUNKS)]
n = wiki.upsert(chunks)
print(f"qdrant transport: {wiki.mode}  |  upserted {n} chunks into '{wiki.collection}'")

embedder: HashEmbedder  dim=384


qdrant transport: memory  |  upserted 7 chunks into 'ir_wiki'


/Users/v843010/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## Retrieval with citations

Hybrid filter by `ticker` + semantic search. Each hit returns its `source_url` so downstream
claims are auditable.

In [3]:
for q in ["gross margin sustainability and competition",
          "supply constraints and lead times",
          "path back to profitability and ROIC"]:
    print(f"\nQUERY: {q}")
    for h in wiki.search(q, ticker="NVDA", k=2, doc_type="transcript"):
        print(f"  [{h['score']:.3f}] ({h['ticker']} {h['period']}/{h['role']}) {h['text'][:80]}...")
        print(f"          source: {h['source_url']}")

# sanity: the margin/competition query should surface the matching analyst question
top = wiki.search("gross margin sustainability and competition", ticker="NVDA", k=1)
assert top and "margin" in top[0]["text"].lower()
print("\nRetrieval grounded + cited.")


QUERY: gross margin sustainability and competition
  [0.466] (NVDA FY2025Q4/analyst) Analyst: Your data center gross margin expanded sharply. How sustainable is this...
          source: https://example.com/nvda/fy25q4#qa1
  [0.191] (NVDA FY2025Q4/exec) CFO: Revenue grew across data center as customers ramped AI training and inferen...
          source: https://example.com/nvda/fy25q4#prep1

QUERY: supply constraints and lead times
  [0.560] (NVDA FY2025Q4/analyst) Analyst: Can you discuss supply constraints and lead times for your latest GPU p...
          source: https://example.com/nvda/fy25q4#qa2
  [0.286] (NVDA FY2025Q4/exec) CFO: Revenue grew across data center as customers ramped AI training and inferen...
          source: https://example.com/nvda/fy25q4#prep1

QUERY: path back to profitability and ROIC
  [0.174] (NVDA FY2025Q4/exec) CFO: Revenue grew across data center as customers ramped AI training and inferen...
          source: https://example.com/nvda/fy25q4#prep1
  [0.

### Switch to a real Qdrant server (docker)
```bash
docker run -p 6333:6333 -p 6334:6334 qdrant/qdrant
```
Then set `QDRANT_MODE=docker` (and `QDRANT_URL`) in `.env`. For real embeddings set
`EMBEDDING_BACKEND=sentence-transformers` (local bge) or `vllm` (MI300X endpoint) — no code change.

**Next (Step 3):** market sentiment from news + social.